ESERCIZIO
Sviluppa una pipeline OCR-NLP in Python
1. Utilizza 'pytesseract' per estrarre il testo da un'immagine di esempio (es. uno screenshot di una citazione testuale)
2. Implementa uan fuzione di pulizia che rimuova ogni carattere non alfanumerico (eccetto gli spazi)
3. Utilizza la libreria 'transformers' per calcolare il sentiment del testo così estratto. 
Suggerimento: se il testo estratto è vuoto o troppo corto, gestisci l'eccezzione evitando l'inferenza del modello NLP

In [ ]:
import os
import re
import torch
import easyocr
from transformers import pipeline

# 1. CONFIGURAZIONE AMBIENTE
os.environ["KERAS_BACKEND"] = "torch"

# Inizializziamo il lettore (scarica i modelli IT/EN alla prima esecuzione)
# È 'semplice' perché non serve configurare percorsi di sistema
print(f"[INFO] Inizializzazione EasyOCR con supporto GPU: {torch.cuda.is_available()}")
reader = easyocr.Reader(['it', 'en'], gpu=torch.cuda.is_available())
print("[INFO] Reader pronto.")

# ------------------------------------------------------------------
# FASE 1 & 2: ESTRAZIONE E PULIZIA (REQUISITI 1 E 2)
# ------------------------------------------------------------------

def extract_and_clean_simple(image_path):
    if not os.path.exists(image_path):
        print(f"[ERRORE] File non trovato: {image_path}")
        return None

    print(f"[OCR] Elaborazione immagine: {os.path.basename(image_path)}...")
    # readtext con detail=0 restituisce solo le stringhe trovate
    results = reader.readtext(image_path, detail=0)
    print(f"[OCR] Trovati {len(results)} frammenti di testo.")
    raw_text = " ".join(results)

    # Pulizia: Rimuoviamo ogni carattere NON alfanumerico (eccetto gli spazi)
    # ^ in regex significa "negazione"
    print("[CLEANUP] Rimozione caratteri speciali e normalizzazione...")
    clean_text = re.sub(r'[^a-zA-Z0-9\s]', '', raw_text)
    
    # Normalizziamo gli spazi bianchi
    clean_text = " ".join(clean_text.split())
    
    print(f"[CLEANUP] Lunghezza testo: {len(raw_text)} -> {len(clean_text)} caratteri.")
    return clean_text

# ------------------------------------------------------------------
# FASE 3: SENTIMENT ANALYSIS (REQUISITO 3)
# ------------------------------------------------------------------

def run_sentiment_analysis(text):
    # GESTIONE ECCEZIONE: Testo vuoto o troppo corto (< 5 caratteri)
    if not text or len(text.strip()) < 5:
        print("[NLP] Errore: Il testo estratto è troppo povero per l'analisi.")
        return None

    print(f"[NLP] Caricamento modello di Sentiment Analysis...")
    
    # Carichiamo la pipeline forzando PyTorch (pt) per evitare conflitti Keras 3
    classifier = pipeline(
        "sentiment-analysis", 
        model="lxyuan/distilbert-base-multilingual-cased-sentiments-student",
        framework="pt",
        device=0 if torch.cuda.is_available() else -1
    )
    
    print(f"[NLP] Esecuzione analisi su: '{text[:50]}...'")
    return classifier(text)[0]

# ------------------------------------------------------------------
# ESECUZIONE
# ------------------------------------------------------------------

if __name__ == "__main__":
    cartella_script = os.path.dirname(os.path.abspath(__file__))
    nome_file = "example_complaint_letter.jpg"
    img_path = os.path.join(cartella_script, nome_file)

    # Simulazione: se il file non esiste, avvisiamo l'utente
    if not os.path.exists(img_path):
        print(f"Immagine non trovata")
    else:
        # 1 & 2: OCR e Pulizia
        testo_pulito = extract_and_clean_simple(img_path)
        print(f"\nTESTO ESTRATTO E PULITO: {testo_pulito}")

        # 3: Sentiment con gestione eccezione
        risultato = run_sentiment_analysis(testo_pulito)

        if risultato:
            print("\n" + "="*50)
            print(f"       RESULTATO ANALISI SENTIMENT")
            print("="*50)
            print(f" TESTO: {testo_pulito[:100]}...")
            print(f" SENTIMENT : {risultato['label'].upper()}")
            print(f" CONFIDENZA: {risultato['score']:.2%}")
            print("="*50)